# Virtual Try-On Pipeline — Complete Setup & Inference

This notebook sets up and runs the full CatVTON virtual try-on pipeline from scratch after cloning the repo.

**Requirements:**
- GPU runtime (T4 / L4 / A100 recommended, minimum ~8 GB VRAM)
- Internet access for downloading model weights

**Supported platforms:** Kaggle, Google Colab, or any Jupyter environment with CUDA GPU.

## 1. Clone Repository & Install Dependencies

In [ ]:
# Clone the repo (skip if already inside it)
import os
if not os.path.exists("model/pipeline.py"):
    !git clone https://github.com/usman9-ai/Virtual-Try-On.git
    os.chdir("Virtual-Try-On")
    print(f"Changed to: {os.getcwd()}")
else:
    print(f"Already in repo: {os.getcwd()}")

In [ ]:
# Install dependencies (compatible versions, avoids huggingface_hub conflict)
!pip install -q torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q accelerate>=0.31.0 transformers>=4.46.0 diffusers>=0.30.0
!pip install -q huggingface_hub>=0.27.0 peft>=0.14.0 safetensors
!pip install -q opencv-python pillow scipy scikit-image tqdm matplotlib
!pip install -q fvcore av cloudpickle omegaconf pycocotools
!pip install -q xformers>=0.0.26

In [ ]:
# Verify GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Enable GPU runtime in your notebook settings.")

## 2. Load the CatVTON Pipeline

In [ ]:
import sys
sys.path.insert(0, os.getcwd())

from model.pipeline import CatVTONPipeline

# Model checkpoint paths
# These are downloaded automatically from HuggingFace Hub
BASE_CKPT = "runwayml/stable-diffusion-inpainting"  # Base SD-inpainting model
ATTN_CKPT = "zhengchong/CatVTON"                    # CatVTON attention adapter
ATTN_CKPT_VERSION = "mix"                            # Options: "mix", "vitonhd", "dresscode"

print("Loading CatVTON pipeline...")
print("(First run will download ~5 GB of model weights)")

pipeline = CatVTONPipeline(
    base_ckpt=BASE_CKPT,
    attn_ckpt=ATTN_CKPT,
    attn_ckpt_version=ATTN_CKPT_VERSION,
    weight_dtype=torch.float16,
    device="cuda",
    skip_safety_check=True,  # Set False in production
    use_tf32=True,
)

print("Pipeline loaded successfully!")

## 3. Prepare Test Images

You need:
- **Person image**: Full-body photo of a person
- **Garment image**: Isolated garment (flat-lay or on model)
- **Mask image** (optional): Binary mask indicating the region to replace

If you don't have a mask, the pipeline can use a full-body mask (replace everything).

In [ ]:
from PIL import Image
import requests
from io import BytesIO
import numpy as np

# ──────────────────────────────────────────────────────────────────────────────
# OPTION A: Load from local files
# ──────────────────────────────────────────────────────────────────────────────
# person_img = Image.open("path/to/person.jpg").convert("RGB")
# garment_img = Image.open("path/to/garment.jpg").convert("RGB")
# mask_img = Image.open("path/to/mask.png").convert("L")  # grayscale binary mask

# ──────────────────────────────────────────────────────────────────────────────
# OPTION B: Load from URLs
# ──────────────────────────────────────────────────────────────────────────────
def load_image_from_url(url):
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")

# ──────────────────────────────────────────────────────────────────────────────
# OPTION C: Use sample images (synthetic test data)
# ──────────────────────────────────────────────────────────────────────────────
# Create simple test images if no real data is available
def create_test_images(height=1024, width=768):
    """Create synthetic test images for pipeline validation."""
    # Person: solid color with simple shape
    person = Image.new("RGB", (width, height), (200, 180, 160))
    # Garment: different color
    garment = Image.new("RGB", (width, height), (50, 100, 200))
    # Mask: upper body region (white = inpaint)
    mask = Image.new("L", (width, height), 0)
    mask_arr = np.array(mask)
    mask_arr[height//4 : 3*height//4, width//4 : 3*width//4] = 255
    mask = Image.fromarray(mask_arr)
    return person, garment, mask

# ═══════════════════════════════════════════════════════════════════════════════
# SELECT YOUR IMAGE SOURCE HERE:
# ═══════════════════════════════════════════════════════════════════════════════

# Uncomment ONE of the following:

# --- Use your own local files ---
# person_img = Image.open("your_person.jpg").convert("RGB")
# garment_img = Image.open("your_garment.jpg").convert("RGB")
# mask_img = Image.open("your_mask.png").convert("L")

# --- Use URLs ---
# person_img = load_image_from_url("https://example.com/person.jpg")
# garment_img = load_image_from_url("https://example.com/garment.jpg")
# mask_img = None  # Will use full mask

# --- Use synthetic test images (default for validation) ---
person_img, garment_img, mask_img = create_test_images()

print(f"Person image:  {person_img.size}")
print(f"Garment image: {garment_img.size}")
print(f"Mask image:    {mask_img.size if mask_img else 'None (will use full mask)'}")

## 4. Run Inference

In [ ]:
# If no mask provided, create a full-body mask (replace entire image)
if mask_img is None:
    mask_img = Image.new("L", person_img.size, 255)

# Inference parameters
HEIGHT = 1024        # Output height (must be divisible by 8)
WIDTH = 768          # Output width (must be divisible by 8)
NUM_STEPS = 24       # Denoising steps (more = better quality, slower)
GUIDANCE_SCALE = 2.5 # CFG scale (higher = more garment fidelity)
SEED = 42            # For reproducibility (set None for random)

# Create generator for reproducibility
generator = torch.Generator(device="cuda").manual_seed(SEED) if SEED else None

print(f"Running inference at {WIDTH}x{HEIGHT}, {NUM_STEPS} steps, guidance={GUIDANCE_SCALE}...")

# Run the pipeline
with torch.cuda.amp.autocast(dtype=torch.float16):
    result_images = pipeline(
        image=person_img,
        condition_image=garment_img,
        mask=mask_img,
        num_inference_steps=NUM_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        height=HEIGHT,
        width=WIDTH,
        generator=generator,
    )

result_image = result_images[0]
print(f"Done! Output size: {result_image.size}")

## 5. Display Results

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(20, 6))

axes[0].imshow(person_img)
axes[0].set_title("Person (Input)")
axes[0].axis("off")

axes[1].imshow(garment_img)
axes[1].set_title("Garment (Condition)")
axes[1].axis("off")

axes[2].imshow(mask_img, cmap="gray")
axes[2].set_title("Mask")
axes[2].axis("off")

axes[3].imshow(result_image)
axes[3].set_title("Try-On Result")
axes[3].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Save the result
output_path = "tryon_result.png"
result_image.save(output_path)
print(f"Result saved to: {output_path}")

## 6. Batch Inference (Multiple Garments)

In [ ]:
def try_on_single(person, garment, mask=None, height=1024, width=768, steps=24, cfg=2.5, seed=None):
    """
    Run a single try-on inference.
    
    Args:
        person: PIL Image or file path of the person
        garment: PIL Image or file path of the garment
        mask: PIL Image (L mode) or None for full-body mask
        height: Output height
        width: Output width  
        steps: Number of denoising steps
        cfg: Classifier-free guidance scale
        seed: Random seed (None for random)
    
    Returns:
        PIL Image result
    """
    if isinstance(person, str):
        person = Image.open(person).convert("RGB")
    if isinstance(garment, str):
        garment = Image.open(garment).convert("RGB")
    if mask is None:
        mask = Image.new("L", person.size, 255)
    elif isinstance(mask, str):
        mask = Image.open(mask).convert("L")
    
    gen = torch.Generator(device="cuda").manual_seed(seed) if seed else None
    
    with torch.cuda.amp.autocast(dtype=torch.float16):
        results = pipeline(
            image=person,
            condition_image=garment,
            mask=mask,
            num_inference_steps=steps,
            guidance_scale=cfg,
            height=height,
            width=width,
            generator=gen,
        )
    return results[0]


# Example: Try multiple garments on the same person
# garment_paths = ["garment1.jpg", "garment2.jpg", "garment3.jpg"]
# results = []
# for gpath in garment_paths:
#     result = try_on_single(person_img, gpath, seed=42)
#     results.append(result)
#     print(f"Completed: {gpath}")

print("Batch helper function ready. Uncomment above to run multiple garments.")

## 7. (Optional) Run with AutoMasker

If you have DensePose and SCHP checkpoints, you can auto-generate masks instead of providing them manually.

In [ ]:
# Optional: AutoMasker (requires DensePose + SCHP checkpoints)
# Uncomment and set paths if you have the models

# DENSEPOSE_CKPT = "./Models/DensePose"  # Path to DensePose checkpoint
# SCHP_CKPT = "./Models/SCHP"            # Path to SCHP checkpoint

# from model.cloth_masker import AutoMasker
# auto_masker = AutoMasker(
#     densepose_ckpt=DENSEPOSE_CKPT,
#     schp_ckpt=SCHP_CKPT,
#     device="cuda",
# )

# # Generate mask automatically
# mask_result = auto_masker(person_img, mask_type="upper")  # "upper", "lower", "overall"
# auto_mask = mask_result["mask"]

# # Run with auto-generated mask
# result = try_on_single(person_img, garment_img, mask=auto_mask)

print("AutoMasker section — uncomment to use if you have DensePose/SCHP models.")

## 8. (Optional) Expose as API with Flask + Ngrok

This creates a public API endpoint you can call from a frontend or mobile app.

In [ ]:
# Install ngrok if running as API
# !pip install -q flask pyngrok

# Uncomment the entire cell below to start the API server
"""
import threading
import io
import base64
from flask import Flask, request, jsonify
from pyngrok import ngrok

app = Flask(__name__)

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok", "gpu": torch.cuda.get_device_name(0)})

@app.route("/tryon", methods=["POST"])
def tryon_endpoint():
    try:
        data = request.json
        
        # Load images from URLs
        person = load_image_from_url(data["person_url"])
        garment = load_image_from_url(data["garment_url"])
        
        # Optional mask URL
        mask = None
        if "mask_url" in data and data["mask_url"]:
            mask = load_image_from_url(data["mask_url"]).convert("L")
        
        # Run inference
        result = try_on_single(
            person, garment, mask=mask,
            height=data.get("height", 1024),
            width=data.get("width", 768),
            steps=data.get("steps", 24),
            cfg=data.get("guidance_scale", 2.5),
            seed=data.get("seed"),
        )
        
        # Encode result as base64
        buffer = io.BytesIO()
        result.save(buffer, format="PNG")
        img_b64 = base64.b64encode(buffer.getvalue()).decode("utf-8")
        
        return jsonify({"success": True, "image_base64": img_b64})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500

# Set your ngrok auth token
# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_TOKEN_HERE"  # <-- Replace this!
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start ngrok tunnel
public_url = ngrok.connect(5000)
print(f"\n{'='*60}")
print(f"Public API URL: {public_url}")
print(f"Health check:   {public_url}/health")
print(f"Try-On POST:    {public_url}/tryon")
print(f"{'='*60}\n")

# Run Flask in a thread (non-blocking)
threading.Thread(target=lambda: app.run(port=5000)).start()
"""

print("API section ready — uncomment the cell content and set NGROK_AUTH_TOKEN to start.")

## 9. (Optional) Training — Fine-tune on Custom Data

If you have paired data (person, garment, ground-truth result), you can fine-tune the model.

In [ ]:
# Training requires a dataset with this structure:
# data/
#   person/   *.jpg — person images
#   garment/  *.jpg — corresponding garment images (same filename)
#   gt/       *.jpg — ground-truth try-on result (same filename)
#   mask/     *.png — binary inpainting mask (same filename, optional)

# To train, run:
# !python train.py \
#     --data_root ./data \
#     --output_dir ./checkpoints/my-finetune \
#     --base_ckpt runwayml/stable-diffusion-inpainting \
#     --attn_ckpt zhengchong/CatVTON \
#     --height 512 \
#     --width 384 \
#     --train_batch_size 1 \
#     --learning_rate 1e-5 \
#     --num_train_epochs 10 \
#     --save_steps 500

# For IUV conditioning (DensePose body-part awareness):
# !python train.py \
#     --data_root ./data \
#     --output_dir ./checkpoints/my-finetune-iuv \
#     --use_iuv_conditioning \
#     --densepose_ckpt ./Models/DensePose

print("Training commands ready — uncomment to run.")
print("Recommended: start at 512x384 for faster iteration, then scale to 1024x768.")

## 10. Cleanup & VRAM Info

In [ ]:
# Check VRAM usage
print(f"VRAM allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"VRAM reserved:  {torch.cuda.memory_reserved() / 1024**3:.2f} GB")
print(f"VRAM total:     {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

# To free memory if needed:
# del pipeline
# torch.cuda.empty_cache()
# print("Pipeline unloaded, VRAM freed.")